In [2]:
import gym_super_mario_bros
from gym.wrappers import GrayScaleObservation
from nes_py.wrappers import JoypadSpace
from stable_baselines3.common.vec_env import VecFrameStack, DummyVecEnv
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
import os
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback

In [3]:
JoypadSpace.reset = lambda self, **kwargs: self.env.reset(**kwargs)
env = gym_super_mario_bros.make('SuperMarioBros-v0', apply_api_compatibility=True, render_mode='human')
env = JoypadSpace(env, SIMPLE_MOVEMENT)
env = GrayScaleObservation(env, keep_dim=True)
env = DummyVecEnv([lambda: env])
env = VecFrameStack(env, 4, channels_order='last')

/home/phile/miniconda3/envs/gym/lib/python3.10/site-packages/gym/envs/registration.py:555: UserWarning: WARN: The environment SuperMarioBros-v0 is out of date. You should consider upgrading to version `v3`.
  logger.warn(
/home/phile/miniconda3/envs/gym/lib/python3.10/site-packages/gym/envs/registration.py:627: UserWarning: WARN: The environment creator metadata doesn't include `render_modes`, contains: ['render.modes', 'video.frames_per_second']
  logger.warn(
/home/phile/miniconda3/envs/gym/lib/python3.10/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


In [4]:
state = env.reset()

In [5]:
class TrainLogCallback(BaseCallback):
    def __init__(self, check_freq, save_path, verbox=1):
        super(TrainLogCallback, self).__init__(verbose=1)
        self.check_freq = check_freq
        self.save_path = save_path

    def _init_callback(self):
        if self.save_path is not None:
            os.makedirs(self.save_path, exist_ok=True)

    def _on_step(self):
        if self.n_calls % self.check_freq == 0:
            model_path = os.path.join(self.save_path, 'best_model_{}'.format(self.n_calls))
            self.model.save(model_path)
        return True

In [6]:
checkpoint_path = './train'
log_dir = './logs'

callback = TrainLogCallback(check_freq=10000, save_path=checkpoint_path)

In [7]:
model = PPO('CnnPolicy', env, verbose=1, tensorboard_log=log_dir, learning_rate=0.000001, n_steps=512)
#model.learn(total_timesteps=1000000, callback=callback)

Using cuda device
Wrapping the env in a VecTransposeImage.


In [8]:
model = PPO.load('./train/best_model_1000000.zip')

In [9]:
state = env.reset()

In [ ]:
state = env.reset()
while True:

    action, _ = model.predict(state)
    state, reward, done, info = env.step(action)
    env.render()

/home/phile/miniconda3/envs/gym/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):
/home/phile/miniconda3/envs/gym/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:272: UserWarning: WARN: No render modes was declared in the environment (env.metadata['render_modes'] is None or not defined), you may have trouble when calling `.render()`.
  logger.warn(
